# 🚀 Pipeline GHEN Digital - Generador de Contenidos Técnicos

Este notebook implementa 3 métodos para generar contenido técnico de calidad:

1. **Por Tópico**: Define un tema técnico amplio y el sistema sugerirá una keyword óptima
2. **Por Keyword**: Usa directamente una keyword específica
3. **Por Newsletter/URL**: Analiza contenido de newsletters tech para generar artículos

---

## 📦 Configuración Inicial

In [1]:
# Importar librerías
import sys
import os
from dotenv import load_dotenv
import pandas as pd
import importlib

# Cargar variables de entorno
load_dotenv()

# Crear directorio outputs si no existe
os.makedirs('outputs', exist_ok=True)

# Importar módulos personalizados
from longcontent_generator import core, scraper, utils, config

# 🔄 Recargar módulos si ya fueron importados (útil después de cambios)
# Forzar recarga de gmail si existe en memoria
if 'longcontent_generator.gmail' in sys.modules:
    import longcontent_generator.gmail
    importlib.reload(longcontent_generator.gmail)
    print("🔄 Módulo gmail recargado")

importlib.reload(core)

print("✅ Librerías importadas correctamente")
print(f"📊 Modelo Gemini: {config.CONFIG['gemini_model']}")
print(f"📁 Outputs en: {config.CONFIG['output_dir']}")
print("🔄 Módulo core recargado")

/Users/gabrielnoguera/Documents/ghen/LongContent_Generator_Script/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Configuración Gemini cargada correctamente
📊 Configuración cargada:
   • Modelo: gemini-2.5-flash
   • Temperatura: 0.7
   • País por defecto: ES
✅ LongContent Generator v2.2.0 cargado correctamente
📋 Funciones principales disponibles:
   • google_custom_search() - Búsqueda en Google
   • scrape_articles_batch() - Scraping de artículos
   • analyze_articles_batch() - Análisis SEO con Gemini
   • generate_article_from_outline() - Generación de contenido
   • qa_article_coverage() - Análisis de calidad
   • publish_article_from_markdown_cleaned() - Publicación WordPress
   • extract_newsletter_from_gmail() - 🆕 Leer newsletters desde Gmail
   • list_gmail_newsletters() - 🆕 Listar newsletters disponibles
   • add_source_links_to_article() - 🆕 Referencias automáticas
   • generate_featured_image_prompt() - 🆕 Generar prompt de imagen
   • generate_image_from_prompt() - 🆕 Generar imagen con Imagen 3
🔄 Módulo gmail recargado
✅ Librerías importadas correctamente
📊 Modelo Gemini: gemini-2.5-fl

In [2]:
# Cargar contexto de GHEN (personalidad técnica)
ghen_context = core.load_ghen_context()  # Ahora funciona!

print("\n📋 Contexto de GHEN cargado:")
print(f"   • Personalidad: {len(ghen_context['personality'])} caracteres")
if ghen_context['project']:
    print(f"   • Proyecto: {len(ghen_context['project'])} caracteres")
if ghen_context['audience']:
    print(f"   • Audiencia: {len(ghen_context['audience'])} caracteres")

✅ Personalidad técnica de GHEN cargada

📋 Contexto de GHEN cargado:
   • Personalidad: 6675 caracteres


---

## 🎯 SELECCIONA TU MÉTODO DE CREACIÓN

Cambia el valor de `metodo_seleccionado` a una de estas opciones:
- `"topico"` - Generar a partir de un tema amplio
- `"keyword"` - Generar a partir de una keyword específica
- `"newsletter"` - Generar a partir de una URL de newsletter

**Ejecuta solo UNA de las secciones según tu elección.**

In [3]:
# 🔧 CONFIGURA AQUÍ TU MÉTODO
metodo_seleccionado = "newsletter"  # Cambia a "topico", "keyword" o "newsletter"

print(f"🎯 Método seleccionado: {metodo_seleccionado.upper()}")

🎯 Método seleccionado: NEWSLETTER


---

# 📝 MÉTODO 1: Generación por Tópico

Define un tema técnico amplio y el sistema sugerirá la mejor keyword para SEO.

In [4]:
if metodo_seleccionado == "topico":
    # Define tu tópico técnico aquí
    topico = "Implementación de agentes ReAct con LangGraph para sistemas de producción"
    
    print(f"📌 Tópico definido: {topico}")
    print("\n🤖 Analizando el tópico y sugiriendo una keyword óptima...\n")
    
    # Sugerir keyword desde el tópico
    keyword_principal = core.suggest_keyword_from_topic(topico, ghen_context)
    
    if keyword_principal:
        print(f"\n✨ Keyword sugerida: '{keyword_principal}'")
        print("\n💡 Puedes continuar con el pipeline usando esta keyword.")
    else:
        print("❌ No se pudo generar una keyword. Verifica la configuración de Gemini.")
else:
    print("⏭️  Método no seleccionado. Pasa a la siguiente sección.")

⏭️  Método no seleccionado. Pasa a la siguiente sección.


---

# 🔑 MÉTODO 2: Generación por Keyword

Define directamente la keyword con la que quieres trabajar.

In [ ]:
if metodo_seleccionado == "keyword":
    # Define tu keyword técnica aquí
    keyword_principal = "mlops best practices for llm deployment"
    
    print(f"🔑 Keyword definida: '{keyword_principal}'")
    print("\n💡 Continuando con el pipeline de investigación...")
else:
    print("⏭️  Método no seleccionado. Pasa a la siguiente sección.")

⏭️  Método no seleccionado. Pasa a la siguiente sección.


---

# 📰 MÉTODO 3: Generación desde Newsletter

Tienes 2 opciones:
- **Opción A**: Pegar URL pública de newsletter
- **Opción B**: Usar Gmail API para leer newsletters de tu bandeja de entrada (🆕)

In [4]:
if metodo_seleccionado == "newsletter":
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # 🔧 ELIGE TU MÉTODO DE ACCESO AL NEWSLETTER
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    
    usar_gmail = True  # True = Gmail API | False = URL pública
    
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    
    newsletter_analysis = None
    
    if usar_gmail:
        print("📧 MÉTODO: Gmail API")
        print("="*70)
        
        # Opción 1: Pegar URL directa de Gmail
        gmail_url = "https://mail.google.com/mail/u/0/?ui=2&ik=9f373130bb&view=lg&permmsgid=msg-f:1849085179733622850"
        
        print(f"📰 URL de Gmail: {gmail_url}")
        print("\n🔍 Extrayendo newsletter desde Gmail...\n")
        
        newsletter_analysis = core.extract_newsletter_from_gmail(gmail_url, ghen_context)
    
    else:
        print("🌐 MÉTODO: URL Pública")
        print("="*70)
        
        # Opción 2: URL pública de newsletter
        newsletter_url = "https://newsletter-tech-url.com/latest"
        
        print(f"📰 URL de newsletter: {newsletter_url}")
        print("\n🔍 Analizando el contenido técnico...\n")
        
        newsletter_analysis = core.extract_and_summarize_url(newsletter_url, ghen_context)
    
    # Mostrar resultados
    if newsletter_analysis:
        print("\n" + "="*70)
        print("📊 ANÁLISIS DE LA NEWSLETTER TÉCNICA")
        print("="*70)
        print(newsletter_analysis['raw_analysis'])
        print("="*70)
        
        # Guardar el análisis
        with open('outputs/analisis_newsletter.md', 'w', encoding='utf-8') as f:
            f.write(newsletter_analysis['raw_analysis'])
        
        print("\n💾 Análisis guardado en 'outputs/analisis_newsletter.md'")
        print("\n💡 Usa los 'Temas Sugeridos' para generar contenido técnico.")
    else:
        print("❌ No se pudo analizar el contenido.")
else:
    print("⏭️  Método no seleccionado. Pasa a la siguiente sección.")

📧 MÉTODO: Gmail API
📰 URL de Gmail: https://mail.google.com/mail/u/0/?ui=2&ik=9f373130bb&view=lg&permmsgid=msg-f:1849085179733622850

🔍 Extrayendo newsletter desde Gmail...

📧 Extrayendo newsletter con ID: 19a9452c91764042
✅ Autenticación exitosa con Gmail API
✅ Autenticación exitosa con Gmail API
✅ Newsletter obtenido: '[AINews] xAI Grok 4.1: #1 in Text Arena, Better Creative Wri...'
   De: AINews <news@smol.ai>
   Contenido: 30438 caracteres (texto), 186533 caracteres (HTML)
✅ Newsletter obtenido: 186533 caracteres
📋 Asunto: [AINews] xAI Grok 4.1: #1 in Text Arena, Better Creative Writing, Fewer Hallucination
📨 De: AINews <news@smol.ai>
✅ Newsletter obtenido: '[AINews] xAI Grok 4.1: #1 in Text Arena, Better Creative Wri...'
   De: AINews <news@smol.ai>
   Contenido: 30438 caracteres (texto), 186533 caracteres (HTML)
✅ Newsletter obtenido: 186533 caracteres
📋 Asunto: [AINews] xAI Grok 4.1: #1 in Text Arena, Better Creative Writing, Fewer Hallucination
📨 De: AINews <news@smol.ai>
✅ Aná

### 📋 Opción: Listar Newsletters Disponibles en Gmail

Ejecuta esta celda para ver tus últimos newsletters y copiar el ID del que quieras usar.

---

# 🔬 PASO 1: Investigación de Keywords (Opcional)

Si elegiste método 1 o 2, puedes hacer scraping de keywords relacionadas.

In [5]:
# Solo ejecutar si NO estás usando el método newsletter
if metodo_seleccionado in ["topico", "keyword"]:
    print(f"🔍 Scrapeando keywords relacionadas con: '{keyword_principal}'")
    
    # Configuración del scraper
    scraper_config = {
        'keyword': keyword_principal,
        'language': 'es',
        'country': 'es',
        'scrape_levels': 1,  # Nivel de profundidad
        'headless': True
    }
    
    # Crear instancia del scraper
    kw_scraper = scraper.GoogleKeywordScraper(**scraper_config)
    
    # Ejecutar scraping (método correcto: scrape(), no run())
    keywords_df = kw_scraper.scrape()
    
    if keywords_df is not None and not keywords_df.empty:
        # Guardar resultados
        keywords_df.to_csv('outputs/keywords_scraped.csv', index=False)
        print(f"\n✅ {len(keywords_df)} keywords encontradas y guardadas")
        print("\n📋 Primeras 10 keywords:")
        print(keywords_df.head(10))
    else:
        print("⚠️  No se encontraron keywords. Continuando sin ellas.")
else:
    print("⏭️  Saltando investigación de keywords (método newsletter).")

⏭️  Saltando investigación de keywords (método newsletter).


---

# 🌐 PASO 2: Búsqueda de Artículos de Referencia

Busca artículos relacionados para usar como contexto.

In [5]:
if metodo_seleccionado in ["topico", "keyword"]:
    print(f"🔎 Buscando artículos sobre: '{keyword_principal}'")
    
    # Buscar en Google Custom Search
    search_results = core.google_custom_search(
        query=keyword_principal,
        country='ES',
        max_results=5
    )
    
    if not search_results.empty:
        search_results.to_csv('outputs/search_results.csv', index=False)
        print(f"\n✅ {len(search_results)} artículos encontrados")
        print("\n📰 Artículos encontrados:")
        for idx, row in search_results.iterrows():
            print(f"   {idx+1}. {row['title']}")
            print(f"      {row['link']}\n")
    else:
        print("⚠️  No se encontraron artículos.")
else:
    print("⏭️  Saltando búsqueda de artículos (método newsletter).")

⏭️  Saltando búsqueda de artículos (método newsletter).


---

# 📚 PASO 3: Scraping de Contenido de Artículos

In [6]:
if metodo_seleccionado in ["topico", "keyword"]:
    if not search_results.empty:
        print("📖 Scrapeando contenido de los artículos...\n")
        
        scraped_content = []
        
        for idx, row in search_results.iterrows():
            url = row['link']
            print(f"   Scrapeando {idx+1}/{len(search_results)}: {row['title'][:60]}...")
            
            content = core.scrape_article(url)
            
            if content and len(content) > 100:
                scraped_content.append({
                    'title': row['title'],
                    'url': url,
                    'content': content[:3000]  # Limitar a 3000 caracteres
                })
        
        # Guardar contenido scrapeado
        scraped_df = pd.DataFrame(scraped_content)
        if not scraped_df.empty:
            scraped_df.to_csv('outputs/scraped_articles.csv', index=False)
            print(f"\n✅ {len(scraped_df)} artículos scrapeados exitosamente")
        else:
            print("⚠️  No se pudo scrapear ningún artículo")
    else:
        print("⚠️  No hay artículos para scrapear")
        scraped_content = []
else:
    print("⏭️  Usando contenido de newsletter como contexto.")
    scraped_content = []

⏭️  Usando contenido de newsletter como contexto.


---

# ✍️ PASO 4: Generación del Artículo Técnico Final

Genera un artículo técnico completo usando toda la información recopilada.

In [5]:
print("🎨 Preparando contexto para generación del artículo técnico...\n")

# Preparar contexto según el método
context_sources = []

if metodo_seleccionado == "newsletter":
    # Usar el análisis del newsletter como contexto
    if newsletter_analysis:
        # IMPORTANTE: Usar TANTO el análisis COMO el contenido original
        context_sources.append(f"# ANÁLISIS DE LA NEWSLETTER TÉCNICA\n\n{newsletter_analysis['raw_analysis']}")
        context_sources.append(f"# CONTENIDO COMPLETO DE LA NEWSLETTER\n\n{newsletter_analysis['original_content']}")
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # 🔧 CONFIGURA AQUÍ LA KEYWORD PARA EL ARTÍCULO
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # Opciones:
        # 1. Dejar vacío ("") para usar keyword por defecto
        # 2. Usar uno de los "Temas Sugeridos" del análisis anterior
        # 3. Definir tu propia keyword personalizada
        
        keyword_personalizada = ""  # ← EDITA AQUÍ
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        
        if not keyword_personalizada:
            keyword_principal = "News: Actualidad y tendencias en IA"
            print("💡 Usando keyword por defecto para artículo de Actualidad:")
            print(f"   '{keyword_principal}'")
        else:
            keyword_principal = keyword_personalizada
            print(f"✅ Usando keyword personalizada: '{keyword_principal}'")
        
        print(f"\n📋 Contexto preparado:")
        print(f"   • Análisis del newsletter: {len(newsletter_analysis['raw_analysis'])} caracteres")
        print(f"   • Contenido original: {len(newsletter_analysis['original_content'])} caracteres")
        print(f"   • Total de contexto: {sum(len(s) for s in context_sources)} caracteres")
    else:
        print("❌ No hay análisis de newsletter disponible")
        keyword_principal = None
else:
    # Usar artículos scrapeados como contexto (métodos "topico" o "keyword")
    if scraped_content:
        for article in scraped_content:
            context_sources.append(f"# {article['title']}\n\n{article['content']}")
        print(f"✅ Usando {len(scraped_content)} artículos como contexto")
    else:
        print("⚠️  No hay artículos scrapeados. Generando solo con la keyword.")

# Generar artículo técnico
if keyword_principal and context_sources:
    print(f"\n🚀 Generando artículo técnico sobre: '{keyword_principal}'\n")
    
    articulo = core.generate_article_with_context(
        keyword=keyword_principal,
        context_sources=context_sources,
        leo_context=ghen_context
    )
    
    if articulo:
        # 🆕 LIMPIAR META-TEXTO del inicio
        print("\n🧹 Limpiando meta-texto del artículo...")
        articulo = core.clean_article_metatext(articulo)
        
        # 🆕 AGREGAR REFERENCIAS AUTOMÁTICAMENTE (filtra tracking URLs)
        print("\n🔗 Agregando referencias a las fuentes...")
        articulo = core.add_source_links_to_article(articulo, context_sources)
        
        # Guardar artículo
        with open('outputs/articulo_ghen_generado.md', 'w', encoding='utf-8') as f:
            f.write(articulo)
        
        print("\n" + "="*70)
        print("📝 ARTÍCULO TÉCNICO GENERADO")
        print("="*70)
        print(articulo[:1000] + "\n...\n")
        print("="*70)
        print(f"\n💾 Artículo completo guardado en 'outputs/articulo_ghen_generado.md'")
        print(f"📊 Longitud: {len(articulo)} caracteres, ~{len(articulo.split())} palabras")
    else:
        print("❌ Error al generar el artículo")
else:
    print("❌ Falta keyword o contexto para generar el artículo")

🎨 Preparando contexto para generación del artículo técnico...

💡 Usando keyword por defecto para artículo de Actualidad:
   'News: Actualidad y tendencias en IA'

📋 Contexto preparado:
   • Análisis del newsletter: 5886 caracteres
   • Contenido original: 186533 caracteres
   • Total de contexto: 192495 caracteres

🚀 Generando artículo técnico sobre: 'News: Actualidad y tendencias en IA'

🎨 Generando artículo de actualidad con contexto LEO...
✅ Artículo generado (14523 caracteres)

🧹 Limpiando meta-texto del artículo...
✅ Meta-texto eliminado del artículo

🔗 Agregando referencias a las fuentes...
✅ Agregadas 4 referencias al artículo

📝 ARTÍCULO TÉCNICO GENERADO
# Grok 4.1 en la Carrera de los LLMs: Análisis Técnico de un Contendiente Serio para Arquitecturas GenAI en Producción

El ecosistema de los Large Language Models (LLMs) no da tregua. Cada semana, las innovaciones se suceden a un ritmo vertiginoso, y mantenerse al día no es solo una cuestión de curiosidad, sino una necesidad es

---

# 📊 Resumen Final

In [6]:
print("="*70)
print("📊 RESUMEN DEL PROCESO")
print("="*70)
print(f"\n🎯 Método utilizado: {metodo_seleccionado.upper()}")
print(f"🔑 Keyword principal: {keyword_principal}")

if metodo_seleccionado in ["topico", "keyword"]:
    print(f"📚 Artículos scrapeados: {len(scraped_content) if scraped_content else 0}")
elif metodo_seleccionado == "newsletter":
    print(f"📰 Newsletter analizada: {'Sí' if newsletter_analysis else 'No'}")

print(f"\n✅ Archivos generados en outputs/:")
for file in ['articulo_ghen_generado.md', 'analisis_newsletter.md', 'keywords_scraped.csv', 
             'search_results.csv', 'scraped_articles.csv']:
    filepath = f'outputs/{file}'
    if os.path.exists(filepath):
        size = os.path.getsize(filepath)
        print(f"   • {file} ({size:,} bytes)")

print("\n" + "="*70)
print("🎉 ¡Proceso completado!")
print("="*70)

📊 RESUMEN DEL PROCESO

🎯 Método utilizado: NEWSLETTER
🔑 Keyword principal: News: Actualidad y tendencias en IA
📰 Newsletter analizada: Sí

✅ Archivos generados en outputs/:
   • articulo_ghen_generado.md (15,181 bytes)
   • analisis_newsletter.md (5,989 bytes)
   • keywords_scraped.csv (201 bytes)
   • search_results.csv (1,324 bytes)
   • scraped_articles.csv (22,414 bytes)

🎉 ¡Proceso completado!


---

# 🎨 PASO 5 (Opcional): Generación de Imagen Destacada

Genera automáticamente una imagen destacada para el artículo usando **Vertex AI Imagen 3**.

**Requisitos previos:**
- Variables de entorno configuradas: `PROJECT_ID`, `LOCATION`, `SERVICE_ACCOUNT_KEY`
- Service account JSON en la raíz del proyecto

**Configuración:**
- `GENERATE_IMAGE`: `True` para generar imagen, `False` para omitir
- Costo: ~$0.04 USD por imagen
- La imagen se guarda en `outputs/featured_image.png`

In [ ]:
# ========================================
# CONFIGURACIÓN: Generación de Imagen
# ========================================

GENERATE_IMAGE = True  # Cambiar a False para omitir generación de imagen

# ========================================
# Generación de imagen destacada
# ========================================

featured_image_path = None

if GENERATE_IMAGE:
    try:
        print("=" * 50)
        print("🎨 GENERANDO IMAGEN DESTACADA")
        print("=" * 50)
        
        # 1. Leer artículo generado
        with open('outputs/articulo_ghen_generado.md', 'r', encoding='utf-8') as f:
            article_text = f.read()
        
        # 2. Generar prompt optimizado con Gemini
        from longcontent_generator import generate_featured_image_prompt
        
        image_prompt = generate_featured_image_prompt(
            article_text=article_text,
            article_title=keyword_principal
        )
        
        if image_prompt:
            # 3. Generar imagen con Vertex AI Imagen 3
            from longcontent_generator import generate_image_from_prompt, optimize_image_for_wordpress
            
            featured_image_path = generate_image_from_prompt(
                prompt=image_prompt,
                output_path="outputs/featured_image.png"
            )
            
            if featured_image_path:
                # 4. Optimizar para WordPress
                featured_image_path = optimize_image_for_wordpress(featured_image_path)
                print(f"\n✅ Imagen destacada lista: {featured_image_path}")
            else:
                print("\n⚠️  No se pudo generar la imagen")
        else:
            print("\n⚠️  No se pudo generar el prompt de imagen")
    
    except Exception as e:
        print(f"\n❌ Error al generar imagen: {e}")
        print("   Continuando sin imagen destacada...")
        featured_image_path = None
else:
    print("⏭️  Generación de imagen omitida (GENERATE_IMAGE = False)")

🎨 GENERANDO IMAGEN DESTACADA
🎨 Generando prompt para imagen destacada...
✅ Prompt generado: Futuristic abstract representation of AI innovation and data trends. A luminous,...
🎨 Generando imagen con Imagen 3...
📝 Prompt: Futuristic abstract representation of AI innovation and data trends. A luminous, interconnected neur...
✅ Prompt generado: Futuristic abstract representation of AI innovation and data trends. A luminous,...
🎨 Generando imagen con Imagen 3...
📝 Prompt: Futuristic abstract representation of AI innovation and data trends. A luminous, interconnected neur...
✅ Imagen generada y guardada en: outputs/featured_image.png
💰 Costo estimado: $0.04 USD
✅ Imagen generada y guardada en: outputs/featured_image.png
💰 Costo estimado: $0.04 USD
✅ Imagen optimizada para WordPress

✅ Imagen destacada lista: outputs/featured_image.png
✅ Imagen optimizada para WordPress

✅ Imagen destacada lista: outputs/featured_image.png


---

# 📰 PASO 6 (Opcional): Publicación en WordPress

Publica el artículo generado directamente en WordPress como borrador o publicado.

In [13]:
# ========================================
# CONFIGURACIÓN: Publicación en WordPress
# ========================================

# IMPORTANTE: Actualiza estos valores antes de publicar
ARTICLE_TITLE = "News: tendencias y actualidad IA"  # Cambiar por el título real
MARKDOWN_FILE = "outputs/articulo_ghen_generado.md"
PUBLICATION_STATUS = "draft"  # Opciones: 'draft' o 'publish'

# Verificar si el archivo existe, si no, usar fallback
import os
if not os.path.exists(MARKDOWN_FILE):
    if os.path.exists("outputs/articulo_completo.md"):
        MARKDOWN_FILE = "outputs/articulo_completo.md"
        print(f"⚠️  {MARKDOWN_FILE} no encontrado, usando outputs/articulo_completo.md")

# NOTA: Las credenciales se cargan automáticamente desde .env
# Necesitas tener configuradas las variables:
#   WORDPRESS_LOGIN_GHEN
#   WORDPRESS_PASSWORD_GHEN

print(f"📝 Título configurado: {ARTICLE_TITLE}")
print(f"📄 Archivo: {MARKDOWN_FILE}")
print(f"📊 Estado: {PUBLICATION_STATUS}")
print(f"🖼️  Imagen destacada: {'Sí' if 'featured_image_path' in dir() and featured_image_path else 'No'}")

# ========================================
# Publicar en WordPress
# ========================================

from longcontent_generator import publish_article_from_markdown_cleaned

print("\n" + "=" * 50)
print("📤 PUBLICANDO EN WORDPRESS")
print("=" * 50)

# Verificar si tenemos imagen destacada generada
image_to_use = None
if 'featured_image_path' in dir() and featured_image_path:
    image_to_use = featured_image_path
    print(f"🖼️  Usando imagen destacada: {featured_image_path}")

# Publicar artículo (con o sin imagen)
result = publish_article_from_markdown_cleaned(
    article_title=ARTICLE_TITLE,
    markdown_file_path=MARKDOWN_FILE,
    status=PUBLICATION_STATUS,
    featured_image_path=image_to_use
)

print("\n" + "=" * 50)
if result and result.get('success'):
    print("✅ PUBLICACIÓN EXITOSA")
    print("=" * 50)
    print(f"📝 Post ID: {result['post_id']}")
    print(f"🔗 URL: {result['post_url']}")
    if result.get('featured_media_id'):
        print(f"🖼️  Imagen destacada asignada (ID: {result['featured_media_id']})")
    print("\n💡 El artículo está en modo '{}'. Ve a WordPress para revisarlo.".format(PUBLICATION_STATUS))
else:
    print("❌ ERROR EN LA PUBLICACIÓN")
    print("=" * 50)
    print("Verifica:")
    print("  • Credenciales en .env (WORDPRESS_LOGIN_GHEN, WORDPRESS_PASSWORD_GHEN)")
    print("  • Conexión a Internet")
    print("  • URL de WordPress (https://ghendigital.com/wp-json/wp/v2/posts)")

📝 Título configurado: News: tendencias y actualidad IA
📄 Archivo: outputs/articulo_ghen_generado.md
📊 Estado: draft
🖼️  Imagen destacada: Sí

📤 PUBLICANDO EN WORDPRESS
🖼️  Usando imagen destacada: outputs/featured_image.png
✅ Archivo Markdown leído: outputs/articulo_ghen_generado.md
✅ Markdown limpiado y corregido automáticamente
✅ Markdown convertido a HTML correctamente
🖼️  Procesando imagen destacada...
📤 Subiendo imagen a WordPress: featured_image.png
✅ Imagen subida correctamente
   🆔 Media ID: 1145
   🔗 URL: https://ghendigital.com/wp-content/uploads/2025/11/featured_image-1.jpg
🔄 Publicando en WordPress (status: draft)...
✅ Imagen subida correctamente
   🆔 Media ID: 1145
   🔗 URL: https://ghendigital.com/wp-content/uploads/2025/11/featured_image-1.jpg
🔄 Publicando en WordPress (status: draft)...
✅ Artículo publicado correctamente
   📝 ID del post: 1146
   🔗 URL: https://ghendigital.com/?p=1146
   🖼️  Imagen destacada asignada (ID: 1145)

✅ PUBLICACIÓN EXITOSA
📝 Post ID: 1146
🔗 U

In [ ]:
# OPCIONAL: Lista newsletters disponibles en tu Gmail
# Útil para encontrar el ID del newsletter que quieres analizar

if metodo_seleccionado == "newsletter":
    print("📧 Listando tus newsletters recientes...\n")
    
    # Puedes filtrar por remitente específico
    sender_filter = None  # e.g., "newsletter@substack.com" o None para todos
    
    newsletters = core.list_gmail_newsletters(
        sender_filter=sender_filter,
        max_results=10
    )
    
    if newsletters:
        print("\n" + "="*70)
        print("📬 NEWSLETTERS DISPONIBLES EN TU GMAIL")
        print("="*70)
        
        for i, nl in enumerate(newsletters, 1):
            print(f"\n{i}. 📨 {nl['subject']}")
            print(f"   De: {nl['from']}")
            print(f"   Fecha: {nl['date']}")
            print(f"   ID: {nl['id']}")
            print(f"   URL: https://mail.google.com/mail/u/0/#inbox/{nl['id']}")
        
        print("\n" + "="*70)
        print("\n💡 Copia el ID o URL del newsletter que quieras usar")
        print("   y pégalo en la celda anterior (variable gmail_url)")
    else:
        print("⚠️  No se encontraron newsletters")
else:
    print("⏭️  Salta esta celda si no usas método newsletter")